In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import shutil

In [15]:
from ultralytics import YOLO
model = YOLO("./last.pt")

In [13]:
from paddleocr import PaddleOCR
import cv2
import re
# test_folder = '/kaggle/working/results'
test_folder = "./results"
# !rm -rf ./results
# os.mkdir(test_folder)
CONFIDENCE_THRESHOLD = 0.5

# https://paddlepaddle.github.io/PaddleOCR/latest/en/quick_start.html#use-by-code
ocr = PaddleOCR(use_angle_cls = True, use_gpu = False)

def paddle_ocr(image, x1, y1, x2, y2):
    image = image[y1:y2, x1: x2]
    # gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    # blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    # sharp = cv2.Laplacian(blurred, cv2.CV_64F)

    # thresh = cv2.adaptiveThreshold(sharp, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
    result = ocr.ocr(image, det=False, rec = True, cls = False)
    text = ""
    for r in result:
        #print("OCR", r)
        scores = r[0][1]
        if np.isnan(scores):
            scores = 0
        else:  # threshold
            scores = int(scores * 100)
        if scores > 60:
            text+=r[0][0]
    pattern = re.compile('[\W]')
    text = pattern.sub('', text)
    text = text.replace("???", "")
    text = text.replace("O", "0")
    text = text.replace("粤", "")
    return str(text)

def predict_ocr(image):
    results = model.predict(image, device='cpu')
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    # plt.imshow(image)
        
    for result in results:
        filtered_boxes = [box for box in result.boxes if box.conf[0] > CONFIDENCE_THRESHOLD]
        for box in filtered_boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            confidence = box.conf[0]  # Get confidence
            
            # Crop bounding box
            # rect = image[y1:y2, x1:x2]
            # plt.imshow(rect)
            # print((x1, y1), (x2, y2))
            text = paddle_ocr(image, x1, y1, x2, y2)
            # print("NONE") if result==None else print(result)
            # text = result[0][0]
                    
            print(f"Detected text: {confidence:.2f}_{text}")
            cv2.putText(image, f'{confidence:.2f}_{text}', (x1-30, y1-10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                    
            cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 3)

            # boxes = [elements[0] for elements in line for line in result]
            # txts = [elements[1][0] for elements in line for line in result]
            # scores = [elements[1][1] for elements in line for line in result]
            # output = draw_ocr(image, boxes, txts, scores, font_path='path_to_font.ttf')
            
            # # text = pytesseract.image_to_string(rect, config='--psm 6')  # perform OCR
    return image

def predict_folder(path_test):
    for img in os.listdir(path_test):
        image = cv2.imread(os.path.join(path_test, img))
        image = predict_ocr(image)
        cv2.imwrite(os.path.join(test_folder, img), image)
        
# predict_folder('/kaggle/input/demo-ttt')

<>:31: SyntaxWarning: invalid escape sequence '\W'
<>:31: SyntaxWarning: invalid escape sequence '\W'
C:\Users\FPT SHOP\AppData\Local\Temp\ipykernel_20996\3680270585.py:31: SyntaxWarning: invalid escape sequence '\W'
  pattern = re.compile('[\W]')


[2025/03/16 03:44:31] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, use_mlu=False, use_gcu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='C:\\Users\\FPT SHOP/.paddleocr/whl\\det\\ch\\ch_PP-OCRv4_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='C:\\Users\\FPT SHOP/.paddleocr/whl\\rec\\ch\\ch_PP-OCRv4_rec_infer', rec_image_inverse=True, rec_image_shape='3, 48,

In [16]:
import cv2
count = 0
# !rm -rf /kaggle/working/vid_results
# os.mkdir("/kaggle/working/vid_results")
video_path = "./data/test.mp4"
vidcap = cv2.VideoCapture(video_path)

fps = int(vidcap.get(cv2.CAP_PROP_FPS))
width = int(vidcap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(vidcap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')  
out = cv2.VideoWriter('./processed_vid.mp4', fourcc, fps, (width, height))

if not vidcap.isOpened():
    print("Error opening video stream")
    exit()

frame_count = 0
while True:
    success, frame = vidcap.read()
    if not success:
        break   
    frame = predict_ocr(frame)
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
    out.write(frame)
    cv2.imwrite(f"./results/frame_{frame_count}.jpg", frame)
    cv2.imshow("Video", frame)
    frame_count += 1
    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

vidcap.release()
out.release()

cv2.destroyAllWindows()


0: 832x1088 7 0s, 609.6ms
Speed: 13.1ms preprocess, 609.6ms inference, 1.2ms postprocess per image at shape (1, 3, 832, 1088)
Detected text: 0.78_

0: 832x1088 4 0s, 463.6ms
Speed: 12.4ms preprocess, 463.6ms inference, 3.3ms postprocess per image at shape (1, 3, 832, 1088)
Detected text: 0.78_

0: 832x1088 4 0s, 437.3ms
Speed: 12.1ms preprocess, 437.3ms inference, 2.0ms postprocess per image at shape (1, 3, 832, 1088)
Detected text: 0.78_

0: 832x1088 5 0s, 425.9ms
Speed: 13.2ms preprocess, 425.9ms inference, 2.1ms postprocess per image at shape (1, 3, 832, 1088)
Detected text: 0.79_

0: 832x1088 4 0s, 431.9ms
Speed: 12.8ms preprocess, 431.9ms inference, 1.8ms postprocess per image at shape (1, 3, 832, 1088)
Detected text: 0.78_

0: 832x1088 4 0s, 429.7ms
Speed: 12.2ms preprocess, 429.7ms inference, 2.2ms postprocess per image at shape (1, 3, 832, 1088)
Detected text: 0.78_

0: 832x1088 5 0s, 426.6ms
Speed: 12.6ms preprocess, 426.6ms inference, 1.6ms postprocess per image at shape (1,

In [5]:
# import cv2
# import math
# videoFile = "./data/vid2.mp4"

# cap = cv2.VideoCapture(videoFile)
# frameRate = cap.get(5) #frame rate
# x=1
# while(cap.isOpened()):
#     frameId = cap.get(1) #current frame number
#     ret, frame = cap.read()
#     if (ret != True):
#         break
#     if (frameId % math.floor(frameRate) == 0):
#         frame = predict_ocr(frame) 
#         cv2.imshow('Video', frame)
#         cv2.waitKey()
#         cv2.destroyAllWindows

# cap.release()
# print ("Done!")